In [2]:
import requests
import time
import os
import re
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
#Readers for request
HEADERS=({"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36","Accept-Language":"en-US, en;q=0.5"})

In [4]:
CATEGORIES = {
    "Mouse": "https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=",
     "StoryBooks": "https://www.flipkart.com/search?q=story+books&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=",
    "BoardGames": "https://www.flipkart.com/search?q=board+games&sid=tng%2Cgfv%2Cisv&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_5_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_1_5_na_na_na&as-pos=1&as-type=RECENT&suggestionId=board+games%7CBoard+Games&requestId=d7cdb4d8-4435-4447-92b9-b42746cba0a4&as-searchtext=board&page=",
    "Shoes": "https://www.flipkart.com/search?q=shoes&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=",
    "Beauty": "https://www.flipkart.com/search?q=beauty&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=",
    "Phones":"h/ttps://www.flipkart.com/search?q=phone&sid=tyy%2C4io&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_3_4_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_3_4_na_na_na&as-pos=3&as-type=RECENT&suggestionId=phone%7CMobiles&requestId=92739d2c-f3fa-478e-92af-33be6c811d45&as-searchtext=phon&page=",
    "Bottle":"https://www.flipkart.com/search?q=Bottle&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=",
    "Laptop":"https://www.flipkart.com/search?q=Laptop&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page="
    "Earphone":"https://www.flipkart.com/search?q=Earphone&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page="
    "lipstick":"https://www.flipkart.com/search?q=lipstick&sid=g9b%2Cffi%2Ctv5%2Cuna&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_3_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_1_3_na_na_na&as-pos=1&as-type=RECENT&suggestionId=lipstick%7CLipstick&requestId=bb323645-b18a-446c-8e14-d387dbc4f8ef&as-searchtext=lip&page="
    "Watch":"https://www.flipkart.com/search?q=watches+for+men&sid=r18%2Cf13&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_5_na_na_na&otracker1=AS_QueryStore_OrganicAutoSuggest_1_5_na_na_na&as-pos=1&as-type=RECENT&suggestionId=watches+for+men%7CWrist+Watches&requestId=17c00e2a-60ff-4ebd-b22b-bb8d8a18cf1b&as-searchtext=watches+for+men&page="
}


flipkart_products = []

In [4]:
import pandas as pd
df=pd.DataFrame(flipkart_products)
df.to_csv("flipkart_products.csv", index=False)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [54]:
df.duplicated().sum()

342

In [ ]:
## Scrapping flipkart using Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import urljoin
import time
options = Options()
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)
Base = "https://www.flipkart.com"
for category,value in CATEGORIES.items():
   print("\nScraping category:", category)
   for page in range(1,20):
       driver.get(value+str(page))
       time.sleep(3)
# Wait until products load
       WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//div[@data-id]")))
# Get all product anchor tags directly
       products = driver.find_elements(By.XPATH, "//div[@data-id]")
       print("Found:", len(products))
       # print(products[0].get_attribute("outerHTML"))
       product_links=[]
       for product in products:
         try:
            link=product.find_element(By.TAG_NAME, "a").get_attribute("href")
            if link:
                product_links.append(link)
         except:
            continue
       for product_url in product_links:
            driver.get(product_url)
            try:
              title = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "span.B_NuCI"))
    ).text
              print(title)
            except:
              print("Title not found")
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//div[@data-id]")))
            anchor = product.find_element(By.XPATH, ".//a[@title]")
            title = anchor.get_attribute("title")       
            raw_reviews = product.find_element(By.CLASS_NAME, "PvbNMB").text
            # clean it → removes parentheses and commas
            clean_reviews = raw_reviews.replace("(", "").replace(")", "").replace(",", "")
            # ----- OFFER PRICE -----
            offer_price = product.find_element(By.CLASS_NAME, "hZ3P6w").text
            offer_price = offer_price.replace("₹", "").replace(",", "").strip()

            # ----- MRP -----
            mrp = product.find_element(By.CLASS_NAME, "kRYCnD").text
            mrp = mrp.replace("₹", "").replace(",", "").strip()

            # ----- DISCOUNT -----
            
            discount = product.find_element(By.CLASS_NAME, "HQe8jr").text
            # get only number % (e.g., "68% off" → "68")
            discount = int("".join(filter(str.isdigit, discount)))
            rating = product.find_element(By.CSS_SELECTOR, "div.MKiFS6").text
    
        
            if title and offer_price:
                      flipkart_products.append({
                          "Title": title,
                          "Offer_price": offer_price,
                          "Real_price":mrp,
                          "Discount" : discount,
                          "Rating": rating,
                          "Review":clean_reviews,
                          "Category": category,
                          "Platform": "Flipkart"
                      })




Scraping category: Mouse


In [55]:
flipkart_products[0]

{'Title': "MAYBELLINE NEW YORK Serum Lipstick (Matte) 8Hr Hydrated color - 002 Maybe It's",
 'Offer_price': '569',
 'Real_price': '599',
 'Discount': 5,
 'Rating': '4.4',
 'Review': '138',
 'Category': 'lipstick',
 'Platform': 'Flipkart'}

In [56]:
df1=pd.read_csv("flipkart_products.csv")
df2=pd.read_csv("flipkart.csv")
Dataframe=pd.concat([df1,df2],ignore_index=True)
Dataframe.to_csv("flipkart.csv",index=False)

In [57]:
DF=pd.read_csv("flipkart.csv")

In [58]:
DF["Category"].value_counts()

Category
lipstick      777
Earphone      748
StoryBooks    745
Bottle        732
Mouse         732
BoardGames    727
Laptop        497
Phones        431
Shoes         410
Beauty        226
Name: count, dtype: int64

In [45]:
len(flipkart_products)

748

In [ ]:
## Scrapping flipkart using Request and beautifulSoup library
from requests_html import HTMLSession
s=HTMLSession()
from urllib.parse import urljoin
Base= "https://www.flipkart.com"
for category,values in CATEGORIES.items():
    # print(f"\n Scrapping category : {category}")
    for i in range(1,20):
       url=values+str(i)
      
       print(f" --> Page {i}")
       # print(f"the url is {url}")
       webpage= s.get(url)
       # print("Status Code:", webpage.status_code)
       print("Final URL:", webpage.url)
       # print(webpage.text[:500])   # pri
       soup = BeautifulSoup(webpage.content,"html.parser")
     
       if soup.title is None:
          # print("Page not loaded properly")
          break
       products = soup.select("div[data-id]")
       # print(len(products))      
       for product in products:
               extract_link=product.select_one("a",{"class":"atJtCj"})
               product_link=extract_link.get("href") if extract_link else None
               if not product_link:
                 continue
               product_url = urljoin(Base,product_link)
               # print(product_url)
               product_response = s.get(product_url, headers=HEADERS)
               # print(product_response.text[:1000])
               product_soup = BeautifulSoup(product_response.text, "html.parser") 
                 # Title
               title_tag = product_soup.select_one("span.B_NuCI")
               title = title_tag.text.strip() if title_tag else None
                     
             #   Price
               price_tag = product_soup.find("div", {"class": "hZ3P6w bnqy13"})
               price = price_tag.text.replace("₹","").replace(",","") if price_tag else None
               
             #   ##Real Price 
               real_price_tag = product_soup.find("div", {"class": "kRYCnD yHYOcc"})
               # real_price = re.sub(r"[^\d]", "", price_tag.get_text())    
               real_price = real_price_tag.text.replace("₹","").replace(",","") if real_price_tag else None
    
             ## Discount
               discount_tag = product_soup.find("span", string=lambda t: t and "% off" in t)
               discount = int("".join(ch for ch in discount_tag.get_text(strip=True) if ch.isdigit())) if discount_tag else 0
               # print(discount)
               # Rating
               rating_tag = product_soup.find("div", {"class": "MKiFS6"})
               rating = rating_tag.text if rating_tag else None
               # print("rating :",rating)
                 ## Reviews
               review_tag=product_soup.find("span",string = lambda t:t and "Ratings" in t)
               review=review_tag.get_text(strip=True).replace("Ratings","").replace(",","") if review_tag else None
               # print("Review :",review)
                 

               if title and price:
                      flipkart_products.append({
                          "Title": title,
                          "Offer_price": price,
                          "Real_price":real_price,
                          # "Total_Payment":o_price,
                          "Discount" : discount,
                          "Rating": rating,
                          "Review":review,
                          "Category": category,
                          "Platform": "Flipkart"
                      })

       

 --> Page 1
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=1
 --> Page 2
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=2
 --> Page 3
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=3
 --> Page 4
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=4
 --> Page 5
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=5
 --> Page 6
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=6
 --> Page 7
Final URL: https://www.flipkart.com/search?q=mouse&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page=7
 --> P